In [1]:
# %pip install langchain_openai langchain_community langchain pymysql chromadb -q

In [1]:
db_user = "root"
db_host = "localhost"
db_password = "lavi9755"
db_name = "classicmodels"

In [2]:
from langchain_community.utilities.sql_database import SQLDatabase
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")


In [3]:
# %pip install google_generativeai

In [4]:

import google.generativeai as genai
import pymysql

from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# Gemini setup
genai.configure(api_key= GOOGLE_API_KEY )
model = genai.GenerativeModel("gemini-2.5-flash")

# DB connection
conn = pymysql.connect(
    host="localhost",
    user="root",
    password= db_password,
    database= db_name
)
cursor = conn.cursor()

c:\Users\LENOVO\OneDrive\Desktop\Langchain\SQLbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print(db.dialect)
print(db.get_usable_table_names)
print(db.table_info)

mysql
<bound method SQLDatabase.get_usable_table_names of <langchain_community.utilities.sql_database.SQLDatabase object at 0x000001E919DDF750>>

CREATE TABLE sales_raw (
	order_id VARCHAR(10), 
	customer_name VARCHAR(100), 
	product_category VARCHAR(50), 
	order_date TEXT, 
	quantity TEXT, 
	price_per_unit TEXT, 
	country VARCHAR(50)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from sales_raw table:
order_id	customer_name	product_category	order_date	quantity	price_per_unit	country
O001	   John Doe   	Electronics	2025-08-10	2	500	India
O002	Jane Smith	HomeAppliance	08/11/2025	3	1200	 United States 
O003	John Doe	Electronics	2025-08-12	two	500	India
*/


In [6]:
# %pip install langchain-google-genai

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [8]:
def get_schema():
    schema = ""
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

    for (table_name,) in tables:
        cursor.execute(f"DESCRIBE {table_name}")
        columns = cursor.fetchall()

        schema += f"\nTable: {table_name}\n"
        for col in columns:
            schema += f"{col[0]} ({col[1]})\n"

    return schema

In [9]:
print(get_schema())


Table: sales_raw
order_id (varchar(10))
customer_name (varchar(100))
product_category (varchar(50))
order_date (text)
quantity (text)
price_per_unit (text)
country (varchar(50))



In [10]:
def generate_query(question):
    schema = get_schema()

    prompt = f"""
You are an expert MySQL developer.

Database schema:
{schema}

Rules:
- Only generate SELECT queries
- Use correct table and column names
- Do not explain anything

Question: {question}

SQL Query:
"""

    response = model.generate_content(prompt)

    query = response.text.strip()
    query = query.replace("```sql", "").replace("```", "").strip()

    return query

In [11]:
def run_query(query):
    if any(word in query.lower() for word in ["drop", "delete", "update", "insert"]):
        return "🚫 Unsafe query blocked"

    try:
        cursor.execute(query)
        return cursor.fetchall()
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
def ask(question):
    query = generate_query(question)
    print("Generated SQL:", query)

    result = run_query(query)

    return result

In [13]:
# %pip install dotenv

In [14]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

for m in genai.list_models():
    print(m.name, m.supported_generation_methods)

models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it ['generateContent', 'countTokens']
models/gemma-3-4b-it ['generateContent', 'countTokens']
models/gemma-3-12b-it ['generateContent', 'countTokens']
models/gemma-3-

In [15]:
print(ask("How many records are there?"))

Generated SQL: SELECT COUNT(*) FROM sales_raw;
((7,),)


In [16]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

answer_prompt = PromptTemplate.from_template(
     """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
 Answer: """
 )

rephrase_answer = answer_prompt | llm | StrOutputParser()

chain = (
     RunnablePassthrough.assign(query=generate_query).assign(
         result=itemgetter("query") | RunnableLambda(run_query)
     )
     | rephrase_answer
 )

chain.invoke({"question": "what product category Jane smith is buying"})


'Jane Smith is buying products from the HomeAppliance category.'

### creating prompt and chain

In [17]:
chain.get_prompts()[0].pretty_print()

Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
 Answer: 


In [18]:
examples = [

    {
        "input": "List all unique customer names after removing extra spaces.",
        "query": "SELECT DISTINCT TRIM(customer_name) FROM orders;"
    },

    {
        "input": "Find all orders where the country is India.",
        "query": "SELECT * FROM orders WHERE TRIM(country) = 'India';"
    },

    {
        "input": "Get all valid orders with proper date format YYYY-MM-DD.",
        "query": "SELECT * FROM orders WHERE order_date REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';"
    },

    {
        "input": "Find total revenue generated from Electronics category.",
        "query": "SELECT SUM(CAST(quantity AS UNSIGNED) * CAST(price AS UNSIGNED)) FROM orders WHERE category = 'Electronics' AND quantity REGEXP '^[0-9]+$';"
    },

    {
        "input": "List orders with missing category.",
        "query": "SELECT * FROM orders WHERE category IS NULL;"
    },

    {
        "input": "Find duplicate orders based on order ID.",
        "query": "SELECT order_id, COUNT(*) FROM orders GROUP BY order_id HAVING COUNT(*) > 1;"
    },

    {
        "input": "Get all orders where quantity is not a number.",
        "query": "SELECT * FROM orders WHERE quantity NOT REGEXP '^[0-9]+$';"
    },

    {
        "input": "Find total number of orders per country after cleaning spaces.",
        "query": "SELECT TRIM(country), COUNT(*) FROM orders GROUP BY TRIM(country);"
    },

    {
        "input": "Get the highest price of any product.",
        "query": "SELECT MAX(CAST(price AS UNSIGNED)) FROM orders;"
    },

    {
        "input": "Find all valid orders where both quantity and price are numeric.",
        "query": "SELECT * FROM orders WHERE quantity REGEXP '^[0-9]+$' AND price REGEXP '^[0-9]+$';"
    }

]

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder,FewShotChatMessagePromptTemplate,PromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
     [
         ("human", "{input}\nSQLQuery:"),
         ("ai", "{query}"),
     ]
 )
few_shot_prompt = FewShotChatMessagePromptTemplate(
     example_prompt=example_prompt,
     examples=examples,
     # input_variables=["input","top_k"],
     input_variables=["input"],
 )
print(few_shot_prompt.format(input1="Get the highest price of any product."))

Human: List all unique customer names after removing extra spaces.
SQLQuery:
AI: SELECT DISTINCT TRIM(customer_name) FROM orders;
Human: Find all orders where the country is India.
SQLQuery:
AI: SELECT * FROM orders WHERE TRIM(country) = 'India';
Human: Get all valid orders with proper date format YYYY-MM-DD.
SQLQuery:
AI: SELECT * FROM orders WHERE order_date REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';
Human: Find total revenue generated from Electronics category.
SQLQuery:
AI: SELECT SUM(CAST(quantity AS UNSIGNED) * CAST(price AS UNSIGNED)) FROM orders WHERE category = 'Electronics' AND quantity REGEXP '^[0-9]+$';
Human: List orders with missing category.
SQLQuery:
AI: SELECT * FROM orders WHERE category IS NULL;
Human: Find duplicate orders based on order ID.
SQLQuery:
AI: SELECT order_id, COUNT(*) FROM orders GROUP BY order_id HAVING COUNT(*) > 1;
Human: Get all orders where quantity is not a number.
SQLQuery:
AI: SELECT * FROM orders WHERE quantity NOT REGEXP '^[0-9]+$';
Human: Find to

In [20]:
## chromo store

In [21]:
# %pip install chromadb

In [22]:
import chromadb

In [24]:
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# correct model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# new clean vector DB
vectorstore = Chroma(collection_name="fresh_examples")

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    vectorstore,
    k=2,
    input_keys=["input"],
)

ValueError: Expected embeddings to be a list of floats or ints, a list of lists, a numpy array, or a list of numpy arrays, got [[-0.032999247312545776, 0.004136873409152031, 0.01806550845503807, -0.05670800432562828, -0.009378037415444851, 0.009525955654680729, 0.0040464336052536964, -0.0014640463050454855, -0.004215278662741184, 0.019520755857229233, 0.006836745422333479, -0.023171987384557724, 0.0035918084904551506, 0.005612850654870272, 0.1352183073759079, -0.026537014171481133, -0.011163050308823586, 0.00808615330606699, 0.005828936118632555, -0.01324350293725729, 0.020253343507647514, -0.002268087351694703, 0.028242381289601326, 0.0118479635566473, -0.005968625657260418, -0.02069731242954731, 0.012866783887147903, 0.00019059216720052063, 0.013442739844322205, -0.005305087193846703, 0.0018447359325364232, 0.01827927678823471, 0.006549761164933443, 0.01188196986913681, 0.014713293872773647, 0.012904064729809761, 0.0017329296097159386, 0.00601794570684433, -0.023878931999206543, 0.02

In [31]:
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma()
# vectorstore.delete_collection()

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,   
    vectorstore,
    k=2,
    input_keys=["input"],
)

# # Test selection
# example_selector.select_examples({"input": "how many employees we have?"})

# # Few-shot prompt
# from langchain_core.prompts import FewShotChatMessagePromptTemplate

# few_shot_prompt = FewShotChatMessagePromptTemplate(
#     example_prompt=example_prompt,
#     example_selector=example_selector,
#     input_variables=["input", "top_k"],
# )

# # Format output
# print(few_shot_prompt.format(input="How many products are there?", top_k=5))

ValueError: Expected embeddings to be a list of floats or ints, a list of lists, a numpy array, or a list of numpy arrays, got [[-0.032999247312545776, 0.004136873409152031, 0.01806550845503807, -0.05670800432562828, -0.009378037415444851, 0.009525955654680729, 0.0040464336052536964, -0.0014640463050454855, -0.004215278662741184, 0.019520755857229233, 0.006836745422333479, -0.023171987384557724, 0.0035918084904551506, 0.005612850654870272, 0.1352183073759079, -0.026537014171481133, -0.011163050308823586, 0.00808615330606699, 0.005828936118632555, -0.01324350293725729, 0.020253343507647514, -0.002268087351694703, 0.028242381289601326, 0.0118479635566473, -0.005968625657260418, -0.02069731242954731, 0.012866783887147903, 0.00019059216720052063, 0.013442739844322205, -0.005305087193846703, 0.0018447359325364232, 0.01827927678823471, 0.006549761164933443, 0.01188196986913681, 0.014713293872773647, 0.012904064729809761, 0.0017329296097159386, 0.00601794570684433, -0.023878931999206543, 0.02

In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
     [
         ("system", "You are a MySQL expert. Given an input question, create a syntactically correct MySQL query to run. Unless otherwise specificed.\n\nHere is the relevant table info: {table_info}\n\nBelow are a number of examples of questions and their corresponding SQL queries."),
         few_shot_prompt,
         ("human", "{input}"),
     ]
 )
print(final_prompt.format(input="How many products are there?",table_info="some table info"))
generate_query = generate_query(llm, db,final_prompt)
chain = (
 RunnablePassthrough.assign(query=generate_query).assign(
     result=itemgetter("query") | run_query
 )
 | rephrase_answer
 )
chain.invoke({"question": "Find duplicate orders based on order ID"})


In [21]:
%pip install pandas

  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/1f/67/af63f83cd6ca603a00fe8530c10a60f0879265b8be00b5930e8e78c5b30b/pandas-3.0.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for tzdata from https://files.pythonhosted.org/packages/c7/b0/003792df09decd6849a5e39c28b513c06e84436a54440380862b5aeff25d/tzdata-2025.3-py2.py3-none-any.whl.metadata
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB 163.8 kB/s eta 0:01:01
   ---------------------------------------- 0.0/9.9 MB 163.8 kB/s eta 0:01:01
   ---------------------------------------- 0.0/9.9 MB 151.3 kB/s eta 0:01:06
   ----------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from operator import itemgetter
from langchain.chains.openai_tools import create_extraction_chain_pydantic
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List
import pandas as pd

def get_table_details():
    # Read the CSV file into a DataFrame
    table_description = pd.read_csv("database_table_descriptions.csv")
    table_docs = []

    # Iterate over the DataFrame rows to create Document objects
    table_details = ""
    for index, row in table_description.iterrows():
        table_details = table_details + "Table Name:" + row['Table'] + "\n" + "Table Description:" + row['Description'] + "\n\n"

    return table_details


class Table(BaseModel):
    """Table in SQL database."""

    name: str = Field(description="Name of table in SQL database.")

table_details = get_table_details()
print(table_details)


In [ ]:
table_details_prompt = f"""Return the names of ALL the SQL tables that MIGHT be relevant to the user question. \
The tables are:

{table_details}

Remember to include ALL POTENTIALLY RELEVANT tables, even if you're not sure that they're needed."""

table_chain = create_extraction_chain_pydantic(Table, llm, system_message=table_details_prompt)
tables = table_chain.invoke({"input": "give me details of customer and their order count"})
tables


In [ ]:
def get_tables(tables: List[Table]) -> List[str]:
    tables  = [table.name for table in tables]
    return tables 

select_table = {"input": itemgetter("question")} | create_extraction_chain_pydantic(Table, llm, system_message=table_details_prompt) | get_tables
select_table.invoke({"question": "give me details of customer and their order count"})


In [ ]:
chain = (
RunnablePassthrough.assign(table_names_to_use=select_table) |
RunnablePassthrough.assign(query=generate_query).assign(
    result=itemgetter("query") | run_query
)
| rephrase_answer
)
chain.invoke({"question": "How many cutomers with order count more than 5"})
